# COSC726 · Lab 1 — LLM Foundations as Measurable Systems

**Agentic Artificial Intelligence · Week 2 · 2-hour guided lab**

Last week you read an agent from the outside. This week you open the **reasoning component** — the LLM —
and treat it as an object of *measurement*, not a magic box. Everything here runs **offline, on the Python
standard library, with no API key**: the model is a deterministic, transparent stand-in so the *mechanisms*
are visible and reproducible on every machine.

Running example throughout: the customer-support agent helping **Layla** with order **#A1032**.

**Six investigations:** tokenization · context budgets · sampling · state · grounding · telemetry.
**Submit:** this notebook executed top-to-bottom, your passing `--self-test`, and six observations.

## 0 · The layer we are studying

A model call maps *constructed context* to *generated output*. It does **not** own state, tools, permissions,
validation, or termination — the application does. Keep this boundary in mind: most bugs you meet this term
live in the harness, not the model.

In [1]:
# Two DELIBERATELY DIFFERENT teaching tokenizers. Neither is authoritative — that is the point.
def greedy_split(text, vocab):
    vocab = sorted(vocab, key=len, reverse=True)
    text, out, i = text.lower(), [], 0
    while i < len(text):
        for v in vocab:
            if v and text.startswith(v.lower(), i):
                out.append(v); i += len(v); break
        else:
            out.append(text[i]); i += 1
    return out

TOK_A = ["order","agent","the","credit","policy","late","1043","ic","ing","ed"," ","-","A","#"]
TOK_B = ["order","ag","ent","the","cred","it","pol","icy","late","10","43","ic","ing","ed"," ","-","A","#"]

for label, vocab in [("A", TOK_A), ("B", TOK_B)]:
    toks = greedy_split("order A-1043", vocab)
    print(f"tokenizer {label}: {len(toks)} tokens  {toks}")

tokenizer A: 5 tokens  ['order', ' ', 'A', '-', '1043']
tokenizer B: 6 tokens  ['order', ' ', 'A', '-', '10', '43']


## 1 · Tokenization is model-specific

The same text, `"order A-1043"`, costs a **different number of tokens** under each tokenizer. There is no
universal "one token ≈ four characters" rule. Below, compare English, Arabic, an identifier, and a JSON
payload — the kinds of input Layla's agent really sees.

In [2]:
samples = {
    "English":    "the late credit policy",
    "Arabic":     "الذكاء الاصطناعي القائم على الوكلاء",   # counted as raw characters by our toy tokenizer
    "Identifier": "customer_order_A-1043",
    "JSON":       '{"status":"pending_approval"}',
}
print(f"{'input':<12}{'tok A':>7}{'tok B':>7}   provenance you MUST record")
print("-" * 60)
for name, text in samples.items():
    a = len(greedy_split(text, TOK_A))
    b = len(greedy_split(text, TOK_B))
    print(f"{name:<12}{a:>7}{b:>7}   (tokenizer + version + the exact text)")
print("\nObservation 1: which input is most expensive, and why? A count WITHOUT")
print("its tokenizer/version is not reproducible evidence.")

input         tok A  tok B   provenance you MUST record
------------------------------------------------------------
English           7      9   (tokenizer + version + the exact text)
Arabic           35     35   (tokenizer + version + the exact text)
Identifier       14     15   (tokenizer + version + the exact text)
JSON             27     27   (tokenizer + version + the exact text)

Observation 1: which input is most expensive, and why? A count WITHOUT
its tokenizer/version is not reproducible evidence.


### Observation 1 — Answer

The **Arabic** input is the most expensive in this teaching tokenizer, at **35 tokens** with both `teach-A` and `teach-B`. The reason is that the toy vocabularies contain no Arabic subword tokens, so the Arabic characters are mostly counted individually. Token cost is tokenizer-specific, so every reported count should include the tokenizer name/version and the exact input text.

**Measured counts:** English = 7/9, Arabic = 35/35, Identifier = 14/15, JSON = 27/27 for `teach-A`/`teach-B`.

## 2 · The context window is a budget — overflow is a policy

The window holds instructions + messages + retrieval + tools + output + reasoning, all at once. When the
input would exceed the budget, real APIs commonly **reject** the request — they do not silently trim. Whether
you drop, summarise, or retrieve is a **policy you choose and log**.

In [3]:
def count_tokens(text, vocab):
    return len(greedy_split(text, vocab))

def prepare_context(messages, context_limit, reserved_output, vocab, strategy="drop_oldest"):
    budget = context_limit - reserved_output
    total = lambda ms: sum(count_tokens(m, vocab) for m in ms)
    if strategy == "reject":
        if total(messages) > budget:
            return {"rejected": True, "reason": f"input {total(messages)} > budget {budget}",
                    "kept": [], "dropped": []}
        return {"rejected": False, "reason": "", "kept": list(messages), "dropped": []}
    kept, dropped = list(messages), []
    while total(kept) > budget and len(kept) > 1:
        dropped.append(kept.pop(1))            # keep messages[0] (system) always
    return {"rejected": total(kept) > budget, "reason": "", "kept": kept, "dropped": dropped}

convo = ["SYSTEM: you are Layla's support agent",
         "USER: where is order #A1032?",
         "ASSISTANT: checking now",
         "USER: it is very late, do I get a credit?"]

print("REJECT policy (tight budget):")
print(" ", prepare_context(convo, context_limit=12, reserved_output=6, vocab=TOK_A, strategy="reject"))
print("\nDROP-OLDEST policy (same messages, logged):")
plan = prepare_context(convo, context_limit=40, reserved_output=6, vocab=TOK_A, strategy="drop_oldest")
print("  kept:   ", plan["kept"])
print("  dropped:", plan["dropped"], " <- LOGGED, never silent")
print("\nObservation 2: for Layla, which message would be UNSAFE to drop silently?")

REJECT policy (tight budget):
  {'rejected': True, 'reason': 'input 111 > budget 6', 'kept': [], 'dropped': []}

DROP-OLDEST policy (same messages, logged):
  kept:    ["SYSTEM: you are Layla's support agent"]
  dropped: ['USER: where is order #A1032?', 'ASSISTANT: checking now', 'USER: it is very late, do I get a credit?']  <- LOGGED, never silent

Observation 2: for Layla, which message would be UNSAFE to drop silently?


### Observation 2 — Answer

It would be unsafe to silently drop the user's message containing the **order ID** and the **credit request**, because the agent could lose information needed to answer correctly or perform an account action. Dropped items must therefore be explicitly logged. The policy must also keep the system message (`messages[0]`).

## 3 · Sampling: temperature changes diversity, not truth

A model outputs a distribution over the next token; a **sampler** picks one. Temperature reshapes the
distribution before picking. At temperature 0 our sampler is **greedy** — the same token wins every time.
That is *repeatability of the sampler*, not reproducibility of the whole system (more in §5).

In [4]:
import math, random

def sample_next(distribution, temperature, rng):
    if temperature <= 0:
        top = max(distribution.values())
        return sorted(k for k, v in distribution.items() if v == top)[0]
    scaled = {k: math.exp(math.log(v) / temperature) for k, v in distribution.items() if v > 0}
    z = sum(scaled.values())
    r, acc = rng.random(), 0.0
    for k in sorted(scaled):
        acc += scaled[k] / z
        if r <= acc:
            return k
    return sorted(scaled)[-1]

dist = {"Paris": 0.82, "London": 0.11, "Lyon": 0.05, "Rome": 0.02}
print("temperature 0 (greedy) over 5 seeds:",
      {sample_next(dist, 0.0, random.Random(s)) for s in range(5)}, " <- always the mode")
print("temperature 1.0 over 8 seeds:      ",
      [sample_next(dist, 1.0, random.Random(s)) for s in range(8)])
print("\nObservation 3: for Layla's ORDER-ID extraction, is high or low temperature safer — and how")
print("would you JUSTIFY the choice with evidence rather than taste?")

temperature 0 (greedy) over 5 seeds: {'Paris'}  <- always the mode
temperature 1.0 over 8 seeds:       ['Paris', 'Lyon', 'Paris', 'Paris', 'Paris', 'Paris', 'Paris', 'Paris']

Observation 3: for Layla's ORDER-ID extraction, is high or low temperature safer — and how
would you JUSTIFY the choice with evidence rather than taste?


### Observation 3 — Answer

**Low temperature, preferably 0, is safer for ORDER-ID extraction.** An order ID is structured data, so consistency and repeatability are more important than creative variation. The choice should be justified with measurable evidence such as extraction accuracy, invalid-ID rate, and repeatability across repeated runs, rather than personal preference. Temperature changes sampling diversity; it does not make an answer more truthful.

## 4 · State lives in the application, not the model

"The model remembers" is shorthand. A bare call is **stateless**; some component (client or provider)
stores the transcript and decides what becomes context. Watch a bare call forget, then a manager remember.

In [5]:
def bare_call(context_messages):
    """Stateless: sees ONLY what it is handed, nothing else."""
    for m in context_messages:
        if "layla" in m.lower():
            return "I can see this is Layla."
    return "I do not know who you are."   # not broken — it was simply handed no history

class ConversationManager:
    """The application component that actually stores and reconstructs state."""
    def __init__(self): self.history = []
    def add(self, msg): self.history.append(msg)
    def call(self): return bare_call(self.history)

print("Bare call, no history handed in:  ", bare_call(["what is my name?"]))
mgr = ConversationManager()
mgr.add("Hi, I'm Layla, about order #A1032")
mgr.add("what is my name?")
print("Managed call, history reconstructed:", mgr.call())
print("\nObservation 4: name the component responsible for retention, deletion, and privacy here.")

Bare call, no history handed in:   I do not know who you are.
Managed call, history reconstructed: I can see this is Layla.

Observation 4: name the component responsible for retention, deletion, and privacy here.


### Observation 4 — Answer

The **ConversationManager** is the application component responsible for retaining and reconstructing conversation history. Therefore, the application layer is also responsible for deciding retention, deletion, and privacy policies for that stored state.

## 5 · Grounding: unsupported generation is an evidence problem

A fluent claim is not a supported one. When an agent may **act** (apply Layla's credit!), an unsupported
fact or an unconfirmed action is unsafe. The fix is a contract: require evidence, or **abstain**.

In [6]:
POLICY_DB = {"late_delivery": {"text": "Orders >3 days late qualify for a 10% credit.",
                               "evidence_id": "policy-late-delivery-v3"}}

def answer_with_grounding(question, require_evidence=True):
    hit = POLICY_DB.get("late_delivery") if "credit" in question or "late" in question else None
    if hit:
        return {"answer": hit["text"], "evidence_id": hit["evidence_id"], "abstained": False}
    if require_evidence:
        return {"answer": "Insufficient evidence — escalating to a human.",
                "evidence_id": None, "abstained": True}
    return {"answer": "Sure, Layla probably gets some money back!", "evidence_id": None, "abstained": False}

print("grounded  :", answer_with_grounding("is Layla's order eligible for a late credit?"))
print("no evidence, abstain :", answer_with_grounding("what is the CEO's home address?", require_evidence=True))
print("no evidence, ungrounded (UNSAFE):", answer_with_grounding("what is the CEO's home address?", require_evidence=False))
print("\nObservation 5: which of these three could safely drive an ACTION on Layla's account?")

grounded  : {'answer': 'Orders >3 days late qualify for a 10% credit.', 'evidence_id': 'policy-late-delivery-v3', 'abstained': False}
no evidence, abstain : {'answer': 'Insufficient evidence — escalating to a human.', 'evidence_id': None, 'abstained': True}
no evidence, ungrounded (UNSAFE): {'answer': 'Sure, Layla probably gets some money back!', 'evidence_id': None, 'abstained': False}

Observation 5: which of these three could safely drive an ACTION on Layla's account?


### Observation 5 — Answer

Only the **grounded** response can safely drive an action on Layla's account because it contains supporting evidence: `policy-late-delivery-v3`. The abstaining response is safe because it refuses to act without evidence, while the ungrounded response is unsafe because it makes an unsupported claim.

## 6 · Telemetry: inspect every model call

A useful trace records enough to estimate cost, reproduce conditions, explain failures, and audit evidence.
This is the normalized response object the course `ModelClient` will standardise (built fully in **Week 4**).

In [7]:
def mock_generate(question):
    """A transparent, deterministic stand-in that returns a NORMALIZED response object."""
    return {
        "provider_model": "course-mock / mock-llm-v2",
        "request_id": "mock-7f31c2",
        "text": "Order #A1032 is 3 days late; a 10% credit applies (see policy-late-delivery-v3).",
        "usage": {"input_tokens": 118, "output_tokens": 36, "cached": 0, "reasoning": 12, "tool": 8},
        "finish_reason": "completed",
        "context_policy": "drop_oldest",
        "dropped_items": ["message-2"],
        "evidence_ids": ["policy-late-delivery-v3"],
        "abstained": False,
        "latency_ms": 640, "retries": 0,
    }

resp = mock_generate("is Layla's order eligible?")
for k, v in resp.items():
    print(f"  {k:<16} {v}")
print("\nObservation 6: which THREE fields would you need to (a) estimate cost, (b) reproduce")
print("the call, and (c) audit the evidence behind the answer?")

  provider_model   course-mock / mock-llm-v2
  request_id       mock-7f31c2
  text             Order #A1032 is 3 days late; a 10% credit applies (see policy-late-delivery-v3).
  usage            {'input_tokens': 118, 'output_tokens': 36, 'cached': 0, 'reasoning': 12, 'tool': 8}
  finish_reason    completed
  context_policy   drop_oldest
  dropped_items    ['message-2']
  evidence_ids     ['policy-late-delivery-v3']
  abstained        False
  latency_ms       640
  retries          0

Observation 6: which THREE fields would you need to (a) estimate cost, (b) reproduce
the call, and (c) audit the evidence behind the answer?


### Observation 6 — Answer

The three fields are:

1. **`usage`** — needed to estimate cost from input/output token usage.
2. **`request_id`** — identifies the specific model call for tracing/reproduction.
3. **`evidence_ids`** — identifies the evidence used to support the generated answer and therefore enables an evidence audit.

For full production-grade reproduction, additional fields such as `provider_model`, context policy, dropped items, and the original request/context would also be important.

## 7 · Wire it up — pass the self-test

Open `COSC726_W02_llm_foundations.py`, implement the three TODOs (`count_tokens`, `prepare_context`,
`sample_next`), then run from a terminal:

```bash
python COSC726_W02_llm_foundations.py --self-test    # target: ALL SELF-TESTS PASSED
```

### Submission checklist
- [ ] This notebook runs top-to-bottom (`Kernel → Restart & Run All`)
- [ ] Your student script prints **ALL SELF-TESTS PASSED**
- [ ] Your **six observations** are written up (one per investigation above)
- [ ] Pushed to your classroom repo under `week02/`

### 🔧 Stretch (optional)
Route **one** approved local or hosted endpoint through a tiny adapter that returns the *same normalized
object* as `mock_generate` — synthetic data only, no real customer text. This previews the provider-adapter
pattern; the full `ModelClient` seam is built in **Lab 3 (Week 4)**.

---
*Next week: prompt & context engineering as behaviour specification — the same model, specified deliberately.*